# 03b — Satellite-Derived Roughness: Alternative Methods

This notebook explores improved approaches for deriving bottom roughness from satellite imagery,
building on the threshold-based classification in notebook 03.

**Methods tested:**
1. **Lyzenga water column correction + Random Forest** on Sentinel-2 (10 m)
2. **PlanetScope multi-temporal compositing + RF** (3 m, seasonal)
3. **Baptist trachytopes** — converting SAV maps to depth-dependent drag for Delft3D FM

**Literature basis:** see `docs/satellite_roughness_review.md` for the full review.

**Key references:**
- Traganos & Reinartz (2018) — Sentinel-2 + Lyzenga for *P. oceanica* / *C. nodosa*
- Poursanidis et al. (2019) — RF + water column correction, Mediterranean
- Zhu et al. (2021) — Delft3D + Baptist vegetation module for seasonal seagrass effects
- Baptist et al. (2007) — analytical drag formulation implemented in Delft3D

## 1. Setup

In [ ]:
%matplotlib inline
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

project_root = Path.cwd().parent
sat_dir = project_root / 'data' / 'raw' / 'satellite'
model_dir = project_root / 'model' / 'dflowfm_v03a'
fig_dir = project_root / 'figures'
os.makedirs(sat_dir, exist_ok=True)

# Stagnone bounding box
LON_MIN, LON_MAX = 12.41, 12.49
LAT_MIN, LAT_MAX = 37.81, 37.92
BBOX = [LON_MIN, LAT_MIN, LON_MAX, LAT_MAX]

# Current roughness map (notebook 03 baseline)
ROUGHNESS_MAP = {0: 0.020, 1: 0.035, 2: 0.050, 3: 0.028}
CLASS_NAMES = {0: 'Bare sand/mud', 1: 'Sparse Cymodocea', 2: 'Dense Posidonia', 3: 'Rock'}

print(f'Project root: {project_root}')
print(f'Model dir:    {model_dir}')

---
## 2. Method 1 — Lyzenga Water Column Correction + Random Forest

The current notebook 03 classifies directly on surface reflectance, which mixes bottom signal
with water column attenuation. The Lyzenga (1978, 2006) depth-invariant index (DII) removes
the water column effect, isolating bottom reflectance.

**Pipeline:**
1. Load Sentinel-2 bands (same scene as nb 03, or multiple)
2. Identify deep-water pixels for attenuation coefficient estimation
3. Compute depth-invariant indices (DII) for band pairs
4. Train Random Forest on DIIs + spectral indices using training polygons
5. Classify and compare with threshold approach

**Ref:** Traganos & Reinartz (2018), Poursanidis et al. (2019)

### 2.1 Load Sentinel-2 bands

Reuses the downloaded scene from notebook 03. If not available, run nb 03 cells 2–3 first.

In [ ]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.crs import CRS
from rasterio.warp import transform_bounds, transform as warp_transform
import glob

def load_band_cropped(filepath, bbox_wgs84):
    """Load a Sentinel-2 band cropped to bounding box."""
    with rasterio.open(filepath) as src:
        left, bottom, right, top = transform_bounds(
            CRS.from_epsg(4326), src.crs, *bbox_wgs84)
        window = from_bounds(left, bottom, right, top, src.transform)
        data = src.read(1, window=window).astype(float) / 10000  # DN to reflectance
        tfm = src.window_transform(window)
        crs = src.crs
    return data, tfm, crs

# Find extracted bands from nb 03
extract_dir = sat_dir / 'extracted'
band_names = ['B02', 'B03', 'B04', 'B05', 'B08']
bands = {}

for bn in band_names:
    matches = glob.glob(str(extract_dir / '**' / f'*_{bn}_*.jp2'), recursive=True)
    res10 = [m for m in matches if '10m' in m]
    res20 = [m for m in matches if '20m' in m]
    path = res10[0] if res10 else (res20[0] if res20 else None)
    if path:
        bands[bn], tfm_utm, crs_utm = load_band_cropped(path, BBOX)
        print(f'  {bn}: {bands[bn].shape} from {os.path.basename(path)}')
    else:
        print(f'  {bn}: NOT FOUND — run notebook 03 cells 2-3 first')

blue, green, red, nir = bands['B02'], bands['B03'], bands['B04'], bands['B08']
print(f'\nAll bands loaded. Shape: {blue.shape}, CRS: {crs_utm}')

### 2.2 Lyzenga Depth-Invariant Index (DII)

The DII for a band pair (i, j) removes depth dependence:

$$DII_{ij} = \ln(L_i) - \frac{k_i}{k_j} \ln(L_j)$$

where $k_i/k_j$ is estimated from the covariance of log-transformed radiance
over a uniform sandy bottom at varying depths (Lyzenga 1978).

In [ ]:
def estimate_ki_kj(band_i, band_j, sand_mask):
    """Estimate attenuation ratio ki/kj from sand pixels at varying depths.
    Uses the covariance method (Lyzenga 1978, Eq. 7).
    """
    xi = np.log(band_i[sand_mask] + 1e-6)
    xj = np.log(band_j[sand_mask] + 1e-6)
    cov_matrix = np.cov(xi, xj)
    var_i = cov_matrix[0, 0]
    var_j = cov_matrix[1, 1]
    cov_ij = cov_matrix[0, 1]
    a = var_i - var_j
    ki_kj = (a + np.sqrt(a**2 + 4 * cov_ij**2)) / (2 * cov_ij)
    return ki_kj


def compute_dii(band_i, band_j, ki_kj):
    """Compute Depth-Invariant Index for band pair."""
    return np.log(band_i + 1e-6) - ki_kj * np.log(band_j + 1e-6)


# --- Identify sand calibration pixels ---
# Criteria calibrated to this scene's reflectance range:
#   blue ~0.12-0.13, green ~0.13, red ~0.12, nir ~0.11, total_rgb ~0.38
ndvi = (nir - red) / (nir + red + 1e-10)
total_refl = blue + green + red
gb_ratio_raw = green / (blue + 1e-10)

# Sand pixels: low vegetation signal, not land, moderate water depth
# - NDVI near zero (bare bottom, no vegetation)
# - GB ratio < 1.05 (vegetation pushes GB > 1.05)
# - NIR not too high (land) or too low (impossible in this scene)
# - Total reflectance in the water range for this scene
sand_mask = (
    (ndvi > -0.05) & (ndvi < 0.02) &     # no vegetation signal
    (gb_ratio_raw < 1.05) &                # not green-dominant (vegetation)
    (nir < 0.15) &                         # not land
    (total_refl > 0.33) & (total_refl < 0.42)  # shallow water range for this scene
)
print(f'Sand calibration pixels: {sand_mask.sum()} ({100*sand_mask.sum()/sand_mask.size:.1f}%)')

if sand_mask.sum() < 100:
    print('WARNING: very few sand pixels found. Relaxing thresholds...')
    sand_mask = (
        (ndvi > -0.08) & (ndvi < 0.03) &
        (gb_ratio_raw < 1.10) &
        (nir < 0.18) &
        (total_refl > 0.32) & (total_refl < 0.50)
    )
    print(f'Relaxed sand pixels: {sand_mask.sum()} ({100*sand_mask.sum()/sand_mask.size:.1f}%)')

# Compute ki/kj ratios for key band pairs
pairs = [
    ('B02', 'B03', blue, green),
    ('B02', 'B04', blue, red),
    ('B03', 'B04', green, red),
]
dii_images = {}
for name_i, name_j, bi, bj in pairs:
    ratio = estimate_ki_kj(bi, bj, sand_mask)
    dii = compute_dii(bi, bj, ratio)
    key = f'DII_{name_i}_{name_j}'
    dii_images[key] = dii
    print(f'  {key}: ki/kj = {ratio:.3f}')

print(f'\n{len(dii_images)} depth-invariant indices computed.')

In [ ]:
# Visualize DIIs
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (key, img) in zip(axes, dii_images.items()):
    vmin, vmax = np.nanpercentile(img[~sand_mask & (total_refl > 0.02)], [2, 98])
    im = ax.imshow(img, extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
                   cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(key)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle('Lyzenga Depth-Invariant Indices — bottom signal isolated from water column')
fig.tight_layout()
plt.show()

### 2.3 Random Forest Classification

Features: 3 DIIs + NDVI + NDAVI + Green/Blue ratio (6 features).

Training data: manually digitized polygons. **TODO**: create training polygons in QGIS
or use the cells below to define rectangular training regions interactively.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

# --- Build feature stack ---
ndavi = (nir - blue) / (nir + blue + 1e-10)
gb_ratio = green / (blue + 1e-10)

feature_names = list(dii_images.keys()) + ['NDVI', 'NDAVI', 'GB_ratio']
feature_stack = np.stack(
    list(dii_images.values()) + [ndvi, ndavi, gb_ratio],
    axis=-1
)
print(f'Feature stack: {feature_stack.shape} ({len(feature_names)} features)')
print(f'Features: {feature_names}')

In [ ]:
# --- Define training regions ---
# Replace these with proper training polygons from QGIS / field knowledge.
# Format: list of (row_slice, col_slice, class_label)
# Coordinates are pixel indices in the cropped image.

# Helper: convert lon/lat to pixel row/col
from rasterio.transform import rowcol
from rasterio.warp import transform as warp_tf

def lonlat_to_rowcol(lon, lat):
    """Convert WGS84 lon/lat to pixel row/col in the cropped image."""
    x_utm, y_utm = warp_tf(CRS.from_epsg(4326), crs_utm, [lon], [lat])
    row, col = rowcol(tfm_utm, x_utm[0], y_utm[0])
    return int(row), int(col)

# Example training regions (adjust based on visual inspection of true color image)
# Each entry: (center_lon, center_lat, half_size_pixels, class)
training_regions = [
    # Bare sand — open areas between Posidonia patches
    (12.430, 37.870, 10, 0),
    (12.460, 37.855, 10, 0),
    (12.445, 37.905, 10, 0),
    # Sparse Cymodocea — shallow areas with light green tones
    (12.450, 37.870, 8, 1),
    (12.440, 37.880, 8, 1),
    # Dense Posidonia — dark patches in true color
    (12.455, 37.865, 8, 2),
    (12.448, 37.875, 8, 2),
    (12.442, 37.860, 8, 2),
    # Rock — near coastline, high reflectance in visible
    (12.425, 37.905, 5, 3),
]

train_X, train_y = [], []
for lon, lat, half_sz, cls in training_regions:
    r, c = lonlat_to_rowcol(lon, lat)
    r0, r1 = max(0, r - half_sz), min(feature_stack.shape[0], r + half_sz)
    c0, c1 = max(0, c - half_sz), min(feature_stack.shape[1], c + half_sz)
    patch = feature_stack[r0:r1, c0:c1].reshape(-1, len(feature_names))
    valid = ~np.any(np.isnan(patch) | np.isinf(patch), axis=1)
    train_X.append(patch[valid])
    train_y.append(np.full(valid.sum(), cls))

train_X = np.vstack(train_X)
train_y = np.concatenate(train_y)
print(f'Training samples: {len(train_y)}')
for cls in sorted(CLASS_NAMES.keys()):
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {(train_y == cls).sum()} samples')

print('\n*** These are PLACEHOLDER training regions. ***')
print('*** Replace with properly digitized polygons for reliable results. ***')

In [ ]:
# --- Train and evaluate Random Forest ---
clf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, train_X, train_y, cv=cv, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

# Train on all data
clf.fit(train_X, train_y)

# Feature importance
print('\nFeature importance:')
for name, imp in sorted(zip(feature_names, clf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:20s}: {imp:.3f}')

In [ ]:
# --- Predict over full image ---
flat_features = feature_stack.reshape(-1, len(feature_names))
valid_mask = ~np.any(np.isnan(flat_features) | np.isinf(flat_features), axis=1)

# Mask land (high NIR) — no deep water mask needed for this shallow lagoon scene
land_mask_flat = (nir.ravel() > 0.20)
predict_mask = valid_mask & ~land_mask_flat

rf_classes = np.full(flat_features.shape[0], -1, dtype=np.int8)
rf_classes[predict_mask] = clf.predict(flat_features[predict_mask])
rf_classes = rf_classes.reshape(blue.shape)

print(f'Classified {predict_mask.sum()} pixels ({100*predict_mask.sum()/predict_mask.size:.1f}%)')
for cls in sorted(CLASS_NAMES.keys()):
    n = (rf_classes == cls).sum()
    print(f'  Class {cls} ({CLASS_NAMES[cls]}): {n} pixels')

In [ ]:
# --- Compare: Threshold (nb 03) vs RF + Lyzenga WCC ---
# Reproduce nb 03 threshold classification (adapted thresholds for this scene)
thresh_classes = np.full(green.shape, 0, dtype=np.int8)
gb = green / (blue + 1e-10)
land_out = nir > 0.20
thresh_classes[(gb > 1.05) & (gb <= 1.25) & ~land_out] = 1
thresh_classes[(gb > 1.25) & ~land_out] = 2
thresh_classes[land_out] = -1

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cmap_class = mcolors.ListedColormap(['#F5DEB3', '#90EE90', '#006400', '#808080'])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, cmap_class.N)
extent = [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]

for ax, data, title in [
    (axes[0], thresh_classes, 'Notebook 03: Threshold (GB ratio)'),
    (axes[1], rf_classes, 'Method 1: RF + Lyzenga WCC'),
]:
    masked = np.ma.masked_where(data == -1, data)
    im = ax.imshow(masked, extent=extent, cmap=cmap_class, norm=norm, aspect='auto')
    ax.set_title(title)
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

cb = plt.colorbar(im, ax=axes, ticks=[0, 1, 2, 3], shrink=0.6, orientation='horizontal', pad=0.08)
cb.set_ticklabels(['Bare sand', 'Sparse Cymodocea', 'Dense Posidonia', 'Rock'])
fig.suptitle('Classification Comparison: Threshold vs Random Forest + Lyzenga WCC', fontsize=13)
fig.tight_layout()
plt.savefig(str(fig_dir / 'roughness_threshold_vs_rf.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Method 2 — PlanetScope Multi-temporal Compositing

PlanetScope (3 m, near-daily) enables:
- Seasonal composites (median, percentiles) that reduce noise and sun glint
- Change detection for seagrass dynamics
- 3× better spatial resolution than Sentinel-2 for delineating Posidonia matte edges

**Requires institutional access to Planet API.**

**Ref:** Wicaksono & Lazuardi (2018), Carlson et al. (2023), Lyons et al. (2020)

In [ ]:
# --- PlanetScope Data Access ---
# Requires a Planet API key (from institutional access)
# Get yours at: https://www.planet.com/account/#/user-settings

PLANET_API_KEY = os.environ.get('PL_API_KEY', '')

if not PLANET_API_KEY:
    print('Planet API key not set.')
    print('Set it as environment variable PL_API_KEY or paste below:')
    # PLANET_API_KEY = 'your-key-here'
else:
    print(f'Planet API key loaded (length: {len(PLANET_API_KEY)})')

In [ ]:
import json
import requests

def search_planet(api_key, bbox, start_date, end_date, cloud_cover=0.1, limit=50):
    """Search PlanetScope catalog for scenes over a bounding box."""
    search_url = 'https://api.planet.com/data/v1/quick-search'
    geojson_bbox = {
        'type': 'Polygon',
        'coordinates': [[
            [bbox[0], bbox[1]], [bbox[2], bbox[1]],
            [bbox[2], bbox[3]], [bbox[0], bbox[3]],
            [bbox[0], bbox[1]]
        ]]
    }
    search_body = {
        'item_types': ['PSScene'],
        'filter': {
            'type': 'AndFilter',
            'config': [
                {'type': 'GeometryFilter', 'field_name': 'geometry', 'config': geojson_bbox},
                {'type': 'DateRangeFilter', 'field_name': 'acquired',
                 'config': {'gte': start_date, 'lte': end_date}},
                {'type': 'RangeFilter', 'field_name': 'cloud_cover',
                 'config': {'lte': cloud_cover}},
                {'type': 'AssetFilter', 'config': ['ortho_analytic_4b_sr']},
            ]
        }
    }
    resp = requests.post(search_url, auth=(api_key, ''), json=search_body)
    resp.raise_for_status()
    features = resp.json().get('features', [])[:limit]
    results = []
    for f in features:
        results.append({
            'id': f['id'],
            'acquired': f['properties']['acquired'],
            'cloud_cover': f['properties']['cloud_cover'],
            'gsd': f['properties'].get('gsd', 3.0),
        })
    return pd.DataFrame(results)


if PLANET_API_KEY:
    ps_scenes = search_planet(
        PLANET_API_KEY, BBOX,
        start_date='2024-06-01T00:00:00Z',
        end_date='2024-09-30T23:59:59Z',
        cloud_cover=0.10,
        limit=50
    )
    print(f'Found {len(ps_scenes)} PlanetScope scenes (cloud < 10%, Jun-Sep 2024)')
    if len(ps_scenes) > 0:
        print(ps_scenes[['id', 'acquired', 'cloud_cover', 'gsd']].head(20).to_string(index=False))
else:
    print('Skipping Planet search — no API key.')
    print('Set PL_API_KEY to enable PlanetScope data access.')

### 3.1 Multi-temporal Compositing Strategy

With ~30-50 cloud-free scenes per summer, we can compute:

| Composite | Description | Use |
|-----------|-------------|-----|
| **Median** | Per-pixel median across all scenes | Removes outliers (sun glint, boats, turbidity events) |
| **10th percentile** | Dark composite | Minimum water surface noise, best bottom signal |
| **Std deviation** | Per-pixel temporal variability | High std → dynamic areas (algae, current zones) |
| **NDVI range** | Max NDVI - Min NDVI | Seasonal seagrass growth signal |

These composites become features for the RF classifier alongside the DIIs.

In [ ]:
# --- Placeholder: Multi-temporal composite computation ---
# This cell will be populated once PlanetScope scenes are downloaded.
# The workflow:
#
# 1. Download N scenes (ortho_analytic_4b_sr = surface reflectance)
# 2. Co-register and crop to Stagnone extent
# 3. Stack into (N, rows, cols, 4) array [Blue, Green, Red, NIR]
# 4. Compute per-pixel composites:
#    median_blue  = np.nanmedian(stack[:, :, :, 0], axis=0)
#    p10_green    = np.nanpercentile(stack[:, :, :, 1], 10, axis=0)
#    std_ndvi     = np.nanstd(ndvi_stack, axis=0)
#    range_ndvi   = np.nanmax(ndvi_stack, axis=0) - np.nanmin(ndvi_stack, axis=0)
# 5. Use composites as additional RF features

print('PlanetScope composite workflow defined.')
print('Requires downloaded scenes — run search cell above with API key first.')
print('\nExpected improvement over single-scene S2:')
print('  - 3x spatial resolution (3m vs 10m)')
print('  - Noise reduction via temporal compositing')
print('  - Seasonal dynamics as classification features')

---
## 4. Method 3 — Baptist Trachytopes for Delft3D FM

Instead of mapping classification → fixed Manning n, the Baptist formulation computes
an effective Chezy roughness that **depends on water depth**:

$$C_b = C_{bed} + \frac{\sqrt{g}}{\kappa} \ln\left(\frac{h}{h_v}\right) \quad \text{for } h > h_v$$

where:
- $C_{bed}$ = bed Chezy (bare sediment)
- $h$ = water depth (varies with tide)
- $h_v$ = vegetation height
- $m$ = stem density (stems/m²)
- $D$ = stem diameter (m)
- $C_D$ = drag coefficient

**Key advantage**: roughness adapts dynamically to tidal water level changes.

**Ref:** Baptist et al. (2007), Zhu et al. (2021), Nepf (2012)

In [ ]:
# --- Vegetation parameters from literature ---
# Sources: Nepf (2012), Zhu et al. (2021), various Med seagrass studies

veg_params = pd.DataFrame([
    {'class': 0, 'name': 'Bare sand/mud',      'h_v': 0.0,  'm': 0,    'D': 0.0,    'C_D': 0.0,
     'manning_static': 0.020, 'notes': 'No vegetation; bed friction only'},
    {'class': 1, 'name': 'Sparse Cymodocea',    'h_v': 0.15, 'm': 800,  'D': 0.004,  'C_D': 1.0,
     'manning_static': 0.035, 'notes': 'C. nodosa: short, flexible, moderate density'},
    {'class': 2, 'name': 'Dense Posidonia',     'h_v': 0.50, 'm': 500,  'D': 0.010,  'C_D': 0.8,
     'manning_static': 0.050, 'notes': 'P. oceanica: tall, dense matte, blade reconfiguration reduces C_D'},
    {'class': 3, 'name': 'Rock/hard bottom',    'h_v': 0.0,  'm': 0,    'D': 0.0,    'C_D': 0.0,
     'manning_static': 0.028, 'notes': 'Elevated roughness from surface texture'},
])

print('Vegetation parameters for Baptist trachytopes:')
print(veg_params[['class', 'name', 'h_v', 'm', 'D', 'C_D', 'manning_static']].to_string(index=False))
print('\nNote: C_D for Posidonia reduced from 1.0 to 0.8 to account for')
print('blade reconfiguration under flow (Nakayama et al., 2020).')

In [ ]:
def baptist_chezy(h, h_v, m, D, C_D, C_bed=45.0, kappa=0.41, g=9.81):
    """Baptist (2007) effective Chezy roughness for submerged vegetation.
    
    Parameters
    ----------
    h : float or array — water depth (m)
    h_v : float — vegetation height (m)
    m : float — stem density (stems/m²)
    D : float — stem diameter (m)
    C_D : float — drag coefficient
    C_bed : float — bed Chezy without vegetation (m^0.5/s)
    """
    if h_v == 0 or m == 0:
        return C_bed * np.ones_like(h, dtype=float)
    
    h = np.asarray(h, dtype=float)
    C_b = np.where(
        h > h_v,
        # Submerged: Baptist Eq. 6
        C_bed / np.sqrt(1 + C_D * m * D * h_v / (2 * g) * C_bed**2) \
            + np.sqrt(g) / kappa * np.log(h / h_v),
        # Emergent: simplified high-drag regime
        C_bed / np.sqrt(1 + C_D * m * D * h / (2 * g) * C_bed**2)
    )
    return C_b


def chezy_to_manning(C, h):
    """Convert Chezy to Manning: n = h^(1/6) / C."""
    return np.where(C > 0, h**(1/6) / C, 0.1)


# --- Demonstrate depth-dependent roughness ---
depths = np.linspace(0.1, 3.0, 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for _, row in veg_params.iterrows():
    C = baptist_chezy(depths, row['h_v'], row['m'], row['D'], row['C_D'])
    n = chezy_to_manning(C, depths)
    ax1.plot(depths, C, label=row['name'], linewidth=2)
    ax2.plot(depths, n, label=row['name'], linewidth=2)
    # Static Manning for reference
    ax2.axhline(row['manning_static'], color=ax2.get_lines()[-1].get_color(),
                linestyle='--', alpha=0.4)

ax1.set_xlabel('Water depth (m)'); ax1.set_ylabel('Effective Chezy (m$^{0.5}$/s)')
ax1.set_title('Baptist: Chezy vs depth'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.set_xlabel('Water depth (m)'); ax2.set_ylabel('Effective Manning n')
ax2.set_title('Baptist: Manning n vs depth (dashed = static from nb 03)')
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_ylim(0, 0.08)

fig.suptitle('Depth-dependent vegetation drag (Baptist formulation)', fontsize=13)
fig.tight_layout()
plt.savefig(str(fig_dir / 'baptist_roughness_vs_depth.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nKey insight: at h=1.0 m (typical Stagnone depth):')
for _, row in veg_params.iterrows():
    C = baptist_chezy(1.0, row['h_v'], row['m'], row['D'], row['C_D'])
    n = chezy_to_manning(C, 1.0)
    print(f'  {row["name"]:20s}: n_baptist={n:.4f}  n_static={row["manning_static"]:.3f}')

### 4.1 Generate Trachytopes Configuration Files

Delft3D FM uses two files for trachytopes:
- `.ttd` — trachytope definitions (vegetation properties per class)
- `.arl` — spatial distribution (which class at each grid cell)

Alternative: use `initialFields.ini` with the Baptist module directly.

In [ ]:
def write_trachytope_def(filepath, veg_params_df):
    """Write a Delft3D trachytope definition file (.ttd).
    
    Format: class_id  formula_nr  param1  param2  param3  param4
    Formula 51 = Baptist: h_v, m*D, C_D, C_bed
    Formula 1  = constant Manning
    """
    with open(filepath, 'w') as f:
        f.write('* Trachytope definitions for Stagnone DT\n')
        f.write('* Generated from satellite-derived seagrass classification\n')
        f.write('* class  formula  param1    param2    param3    param4\n')
        for _, row in veg_params_df.iterrows():
            cls = int(row['class'])
            if row['h_v'] > 0 and row['m'] > 0:
                # Baptist formula (type 51 in Delft3D)
                nD = row['m'] * row['D']  # stem density * diameter
                f.write(f'  {cls}    51    {row["h_v"]:.3f}    {nD:.3f}    {row["C_D"]:.2f}    45.0\n')
            else:
                # Constant Manning (type 1)
                f.write(f'  {cls}     1    {row["manning_static"]:.4f}\n')
    print(f'Written: {filepath}')


# Generate .ttd file
ttd_path = model_dir / 'trachytopes.ttd'
write_trachytope_def(ttd_path, veg_params)

print('\nTo activate in the MDU:')
print('  [Trachytopes]')
print('  trtRou = Y')
print('  trtDef = trachytopes.ttd')
print('  trtL   = trachytopes.arl')
print('  dtTrt  = 600.0')

---
## 5. Summary and Next Steps

| Method | Status | Accuracy | Key Advantage |
|--------|--------|----------|---------------|
| Notebook 03 threshold | Baseline | ~72% (estimated) | No training data needed |
| **Method 1: RF + Lyzenga WCC** | Prototype ready | ~82-90% (literature) | Water column removed, ML-based |
| **Method 2: PlanetScope composite** | Awaiting data | Expected 85%+ | 3m resolution, seasonal dynamics |
| **Method 3: Baptist trachytopes** | Config ready | N/A (physics-based) | Depth-dependent drag, tide-adaptive |

### Immediate actions:
1. Digitize training polygons in QGIS (replace placeholder regions in cell 2.3)
2. Request PlanetScope time series (Jun-Sep 2024) via institutional access
3. Activate existing roughness in v03a (`initialFields.ini`) as baseline test
4. Run v03a with trachytopes `.ttd` as sensitivity experiment

### Publication potential:
- **Gap identified in literature**: no paper completes the full pipeline
  satellite image → friction map → validated 3D hydrodynamic simulation
- Stagnone is ideal: shallow, clear water, well-studied seagrass species
- Target journal: Remote Sensing of Environment or Ecological Modelling